# Notebook 08 — Cross-Validation and Model Selection

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 08 of 15  
**Time estimate:** 75 minutes

> Model selection is where most published biological ML papers get it wrong.
> Cross-validation done correctly is the difference between a publishable result
> and a misleadingly optimistic one.

---
## Step 1 — Motivation

You trained 10 models with different hyperparameters. You pick the one with the best
test performance. You report that performance. This is wrong — you've used the test
set for model selection, not just evaluation. Cross-validation exists to prevent this.
Nested CV exists to prevent the same mistake at the hyperparameter-selection level.

---
## Step 2 — Intuition

**Holdout (train/test split):** Fast but high variance — which samples end up in
test set strongly affects the estimate. Never use for model selection.

**k-fold CV:** Split data into $k$ folds. For each fold: train on remaining $k-1$ folds,
evaluate on held-out fold. Average $k$ performance estimates.
- Reduces variance vs. holdout by averaging
- $k=5$ or $k=10$ standard; $k=n$ (LOOCV) for very small datasets

**Stratified k-fold:** Preserve class proportions in each fold — critical for
imbalanced datasets.

**Nested CV:** Two levels of CV:
- **Outer loop:** Estimate generalization performance
- **Inner loop:** Select hyperparameters

Without nesting: hyperparameter selection on the same data used for performance
estimation → optimistically biased (the model that happened to work well on THIS
data is selected).

**Biological pitfall — patient-level correlation:**
If a study has 3 samples per patient (tumor, normal, metastasis), a random split
may put 2 of those in train and 1 in test. The model "memorizes" the patient —
performance is inflated. Always split at the **patient level**, not the sample level.

---
## Step 3 — Biological Background

**Batch effects and CV:**
If samples from Lab A are all in the training set and samples from Lab B are in test,
the model will fail on Lab B data — but CV within a single batch won't catch this.
Solution: split by batch (lab, experiment run) for evaluation.

**Temporal splits:** For clinical ML, always consider a chronological split:
train on earlier patients, test on later patients. Random CV overestimates performance
for temporal prediction tasks.

**Learning curves:** Plot train and validation accuracy vs. training set size.
- Large gap (train high, val low): high variance → more data or regularization
- Converged (both plateau): high bias → better model or features
- Both still increasing: needs more data

**Repeated k-fold:** Run k-fold multiple times with different random splits,
average results. Reduces variance of CV estimate but multiplies computation.
Standard in published genomics ML papers: 5×5-fold or 10×10-fold CV.

---
## Step 4 — Mathematical Explanation

**k-fold CV estimate of generalization error:**
$$\hat{\text{Err}}_{\text{CV}} = \frac{1}{n} \sum_{k=1}^K \sum_{i \in \mathcal{F}_k} L(y_i, \hat{f}^{-k}(x_i))$$

where $\hat{f}^{-k}$ is the model trained on all data except fold $k$.

**Bias of CV:**
- The model trained on $n(1-1/k)$ samples slightly underestimates performance
  of a model trained on all $n$ samples (pessimistic bias increases with small $k$)
- For $k=n$ (LOOCV): essentially unbiased but high variance
- For $k=10$: good bias-variance balance in practice

**Nested CV:**
```
For fold i in outer CV (k_outer folds):
  Train_i = all data except fold i
  For each hyperparameter config h:
    For fold j in inner CV (k_inner folds) on Train_i:
      Fit model(h) on Train_i minus fold j
      Evaluate on fold j
    val_score(h) = mean over inner folds
  h* = argmax val_score(h)
  Fit model(h*) on all of Train_i
  outer_score[i] = score on fold i
Performance estimate = mean(outer_score)
```

**One standard error rule:** When multiple models have similar performance,
choose the simplest model within 1 SE of the best performance.
Principle of parsimony: simpler models generalize better.

In [ ]:
# Step 6 — Python: CV and model selection from scratch + sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score,
                                       GridSearchCV, learning_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ---- k-fold CV from scratch ----
def stratified_kfold_scratch(y, k=5, seed=42):
    """
    Stratified k-fold cross-validation split.
    Returns: list of (train_idx, val_idx) tuples
    """
    rng = np.random.default_rng(seed)
    n = len(y)
    classes = np.unique(y)
    folds = [[] for _ in range(k)]
    for c in classes:
        c_idx = np.where(y == c)[0]
        c_idx = rng.permutation(c_idx)
        for fold_idx, chunk in enumerate(np.array_split(c_idx, k)):
            folds[fold_idx].extend(chunk.tolist())
    splits = []
    for i in range(k):
        val_idx = np.array(folds[i])
        train_idx = np.array([idx for j in range(k) if j != i for idx in folds[j]])
        splits.append((train_idx, val_idx))
    return splits

# Generate dataset: 500 cancer samples, 5 subtypes, 2000 features → top 50 by variance
rng = np.random.default_rng(42)
n_samples, n_features, n_classes = 500, 200, 5
X = rng.standard_normal((n_samples, n_features))
# Plant signal: first 20 features have class-specific mean shifts
for c in range(n_classes):
    idx = y_idx = np.where(np.arange(n_samples) % n_classes == c)[0]
    X[np.ix_(idx, range(c*4, c*4+4))] += 3.0
y = np.arange(n_samples) % n_classes
print(f'Dataset: {n_samples} samples, {n_features} features, {n_classes} classes')
print(f'Class distribution: {np.bincount(y)}')

# ---- Demonstrate data leakage in CV ----
from sklearn.feature_selection import SelectKBest, f_classif

# LEAKY: Select features before CV
selector = SelectKBest(f_classif, k=20)
X_selected = selector.fit_transform(X, y)  # uses all data — LEAKY
leaky_scores = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    X_selected, y, cv=StratifiedKFold(5))

# CORRECT: Feature selection inside Pipeline (inside CV)
correct_pipe = make_pipeline(
    SelectKBest(f_classif, k=20),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)
correct_scores = cross_val_score(correct_pipe, X, y, cv=StratifiedKFold(5))

print(f'\nLeaky CV: {leaky_scores.mean():.3f} ± {leaky_scores.std():.3f}')
print(f'Correct CV: {correct_scores.mean():.3f} ± {correct_scores.std():.3f}')
print(f'Inflation from leakage: {leaky_scores.mean() - correct_scores.mean():.3f}')

# ---- Nested CV for honest evaluation ----
# Outer: 5-fold; Inner: 3-fold for C hyperparameter selection
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {'logisticregression__C': [0.01, 0.1, 1, 10]}

nested_scores = []
X_std = StandardScaler().fit_transform(X)  # for simplicity (ok here since no feature selection)
for train_idx, test_idx in outer_cv.split(X_std, y):
    X_tr_fold, X_te_fold = X_std[train_idx], X_std[test_idx]
    y_tr_fold, y_te_fold = y[train_idx], y[test_idx]
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
    gs = GridSearchCV(LogisticRegression(max_iter=2000), {'C': [0.01, 0.1, 1, 10]},
                        cv=inner_cv, scoring='accuracy')
    gs.fit(X_tr_fold, y_tr_fold)
    nested_scores.append(gs.score(X_te_fold, y_te_fold))
nested_scores = np.array(nested_scores)
print(f'\nNested CV: {nested_scores.mean():.3f} ± {nested_scores.std():.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Leaky vs. correct CV comparison
ax = axes[0]
bars = ax.bar(['Leaky CV\n(feature sel before split)',
                  'Correct CV\n(Pipeline)'],
                 [leaky_scores.mean(), correct_scores.mean()],
                 color=['tomato', 'steelblue'], edgecolor='black', width=0.5,
                 yerr=[leaky_scores.std(), correct_scores.std()], capsize=5)
ax.axhline(1/n_classes, color='grey', ls='--', lw=1, label=f'Random baseline ({1/n_classes:.2f})')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('A. Data leakage effect\n(leaky feature selection inflates CV)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

# Panel B: Learning curve
ax = axes[1]
pipe_lc = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000))
train_sizes, train_scores, val_scores = learning_curve(
    pipe_lc, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy', random_state=42)
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)
ax.plot(train_sizes, train_mean, 'tomato', lw=2, label='Train')
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color='tomato')
ax.plot(train_sizes, val_mean, 'steelblue', lw=2, label='Validation (5-fold CV)')
ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color='steelblue')
ax.set_xlabel('Training set size')
ax.set_ylabel('Accuracy')
ax.set_title('B. Learning curve\n(converging train/val = needs better model)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)

# Panel C: CV fold-by-fold variation
ax = axes[2]
cv_detailed = StratifiedKFold(5, shuffle=True, random_state=42)
fold_scores_lr = []
fold_scores_rf = []
pipe_lr = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000))
pipe_rf = make_pipeline(RandomForestClassifier(n_estimators=50, random_state=42))
for fold_i, (tr, te) in enumerate(cv_detailed.split(X, y)):
    pipe_lr.fit(X[tr], y[tr]); fold_scores_lr.append(pipe_lr.score(X[te], y[te]))
    pipe_rf.fit(X[tr], y[tr]); fold_scores_rf.append(pipe_rf.score(X[te], y[te]))
fold_n = np.arange(1, 6)
ax.plot(fold_n, fold_scores_lr, 'steelblue', marker='o', lw=1.5, label=f'LogReg mean={np.mean(fold_scores_lr):.3f}')
ax.plot(fold_n, fold_scores_rf, 'tomato', marker='s', lw=1.5, label=f'RF mean={np.mean(fold_scores_rf):.3f}')
ax.axhline(np.mean(fold_scores_lr), color='steelblue', ls='--', lw=0.8, alpha=0.5)
ax.axhline(np.mean(fold_scores_rf), color='tomato', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('C. Fold-by-fold CV scores\n(variance shows reliability of estimate)')
ax.legend(fontsize=8)
ax.set_xticks(fold_n)

plt.suptitle('Module 13 NB08: Cross-Validation and Model Selection', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement stratified k-fold CV from scratch and verify the output matches
   `sklearn.model_selection.StratifiedKFold` on the cancer dataset.
2. Create a scenario with 10 patients, 5 samples each. Split at the patient level
   (GroupKFold) vs. sample level (StratifiedKFold). Show the performance difference.
3. Compare 5-fold, 10-fold, and LOOCV on a 50-sample dataset. How do the variance
   and computational cost differ?
4. Implement nested CV fully from scratch (no sklearn GridSearchCV). Verify it
   produces the same outer scores as the sklearn version.

---
## Step 10 — Quiz

1. What is the difference between 5-fold and leave-one-out CV in terms of bias and variance?
2. What is nested CV and why is it needed?
3. Why is stratified k-fold important for imbalanced datasets?
4. Describe one biological scenario where random CV gives misleadingly optimistic estimates.
5. What is the one standard error rule for model selection?

---
## Step 12 — Reflection

> *[In one paragraph: explain why using the test set multiple times for model
> selection is problematic, using a probability argument about how many random
> models would perform well on a test set just by chance.]*

---
*Next: `09_evaluation_metrics_for_imbalanced_data.ipynb`*